In [1]:
# BigAlpha 2026 端到端提交 —— 推理入口
#
# 平台 import 本 notebook 的 main(datasources, start_date, end_date)，
# 返回 ['date', 'instrument', 'score']。
#
# 配套文件（与本文件平铺在同一目录）：
#   bigalpha_train_local.py   训练与推理共用定义 + 本地训练入口（供复现与审核）
#   bigalpha_model.json       本地训练产物，2 个种子的权重
#
# 训练区间与训练表在 bigalpha_train_local.py 中写死，绝不使用平台注入的
# start_date / end_date 训练。

# -*- coding: utf-8 -*-
"""BigAlpha 2026 端到端提交 —— 推理侧。

平台 import 本文件的 `main(datasources, start_date, end_date)`，返回
['date', 'instrument', 'score']。定义与 `bigalpha_train_local.py` 共用，训练与推理
走同一个 `to_canonical` + `build_windows`，不存在两边漂移。

取数按股票分块，读一批、打一批分、只留三列结果，峰值内存由 CHUNK_SIZE 决定而与
区间长度、股票总数无关 —— 官方 OOM 说明里指出崩溃几乎都发生在这一步。

作为 notebook 提交时，把本文件内容粘进一个 cell 即可（或 `from bigalpha_predict
import main`）。
"""
import os

import numpy as np
import pandas as pd
import torch

from bigalpha_train_local import (CHUNK_SIZE, CLOUD_KEY, MODEL_PATH, PatchEncoder,
                                  build_windows, cloud_frames, load_model)

PREDICT_BATCH = 4096
# 未来函数检测的做法是：同一起始日、把结束日截到中间某天再跑一遍，比对截断日之前
# 的每一格。所以推理必须逐格可复现。分块取数会让不同运行的 batch 形状不同，bf16
# 混精与 TF32 下矩阵乘的求和顺序随之变化，末位抖动足以让两只接近平手的股票排名
# 互换——在 pct 排名上就是 1/N 的整格跳变。因此推理走 fp32、关 TF32，并把原始分
# 舍入到 SCORE_DECIMALS 位再排名，让末位噪声落进同一个值。
SCORE_DECIMALS = 6


def _load_members(device):
    if not os.path.exists(MODEL_PATH):
        raise RuntimeError(
            "找不到 %s。本方案为「本地训练 + 上传权重」，请把训练产物与代码一并提交；"
            "若要在平台重训，改调 bigalpha_train_local.train_and_save()" % MODEL_PATH)
    ckpt = load_model(MODEL_PATH, map_location="cpu")
    models = []
    for sd in ckpt["members"]:
        m = PatchEncoder(**ckpt["model_cfg"]).to(device)
        m.load_state_dict(sd)
        m.eval()
        models.append(m)
    return models, ckpt


def _score(models, X, device):
    """同一批窗口过每个成员，返回 (成员数, 样本数) 的原始分。fp32、不启用混精。"""
    Xt = torch.from_numpy(X)
    out = np.empty((len(models), len(X)), dtype=np.float64)
    with torch.no_grad():
        for mi, model in enumerate(models):
            chunks = []
            for i in range(0, len(Xt), PREDICT_BATCH):
                xb = Xt[i:i + PREDICT_BATCH]
                # 每批补齐到固定行数再算。样本之间本无交互，唯一让两次运行产生
                # 差异的就是尾批形状不同导致 cuBLAS 选到不同的核；形状固定后
                # 逐格结果按位一致，未来函数检测才过得去。
                pad = PREDICT_BATCH - xb.shape[0]
                if pad:
                    xb = torch.cat([xb, xb.new_zeros((pad,) + xb.shape[1:])])
                y = model(xb.to(device)).double().cpu().numpy()
                chunks.append(y[:PREDICT_BATCH - pad] if pad else y)
            out[mi] = np.round(np.concatenate(chunks), SCORE_DECIMALS)
    return out


def main(datasources, start_date, end_date):
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.backends.cudnn.benchmark = False
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    table = datasources.get(CLOUD_KEY) or next(iter(datasources.values()))
    models, ckpt = _load_members(device)
    print("[main] table=%s device=%s members=%d" % (table, device, len(models)), flush=True)

    parts = []
    for canon in cloud_frames(table, start_date, end_date, CHUNK_SIZE):
        X, idx = build_windows(canon, start_date, end_date, "infer")
        del canon
        if not len(idx):
            continue
        raw = _score(models, X, device)
        del X
        # 成员按日内排名平均：不同种子的原始输出尺度不同，直接平均会被尺度大的支配。
        # 这一批的股票是全市场的一个子集，但排名在同一天内做，与分块无关的量只有
        # 相对次序 —— 所以先累积原始分，等所有批次到齐后再按日排名。
        for mi in range(raw.shape[0]):
            idx["m%d" % mi] = raw[mi]
        parts.append(idx.drop(columns=["label"]))
        del raw, idx

    if not parts:
        raise RuntimeError("推理区间内没有可用样本: %s ~ %s" % (start_date, end_date))
    res = pd.concat(parts, ignore_index=True)
    del parts

    mcols = [c for c in res.columns if c.startswith("m") and c[1:].isdigit()]
    ranks = np.column_stack([res.groupby("date")[c].rank(pct=True).to_numpy()
                             for c in mcols])
    res["score"] = ranks.mean(axis=1)
    res = res.rename(columns={"key": "instrument"})
    res = (res.replace([np.inf, -np.inf], np.nan).dropna(subset=["score"])
              .drop_duplicates(["date", "instrument"]).reset_index(drop=True))
    print("[main] rows=%d days=%d instruments=%d"
          % (len(res), res["date"].nunique(), res["instrument"].nunique()), flush=True)
    return res[["date", "instrument", "score"]]


if __name__ == "__main__":
    # 本地自检：把 datasources 指向云端表名，只有在平台上才跑得通
    from bigalpha_train_local import LOCAL_TABLE
    print(main({CLOUD_KEY: "bigalpha_2026_stock_%s" % LOCAL_TABLE},
               "2024-01-01", "2024-03-31").head())


/opt/pyenv/versions/3.11.8/lib/python3.11/site-packages/torch/nn/modules/transformer.py:306: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


[main] table=bigalpha_2026_stock_bar30m device=cpu members=2
[cloud_frames] 字段映射: []
